# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [49]:
# Use this cell to write your code for Task 1
import pandas as pd

df = pd.read_csv('house_sales.csv')

# Treat whitespace-only and common placeholder tokens as missing
city = df["city"].astype(str)

# normalize whitespace
city_stripped = city.str.strip()

placeholder_tokens = {"na", "n/a", "none", "null", "-"}

missing_mask = (
    df["city"].isna() |                      # real NaN
    (city_stripped == "") |                  # empty after trimming
    (city_stripped.str.lower().isin(placeholder_tokens))  # common placeholders
)

missing_city = int(missing_mask.sum())
print(missing_city)

0


# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [50]:
# Use this cell to write your code for Task 2
import pandas as pd
import numpy as np

df = pd.read_csv('house_sales.csv')

df['city'] = df['city'].fillna("Unknown")
df = df.dropna(subset=['sale_price'])

df['sale_date'] = df['sale_date'].fillna('2023-01-01')

df['months_listed'] = pd.to_numeric(df['months_listed'], errors='coerce')
months_mean = round(df['months_listed'].mean(), 1)
df['months_listed'] = df['months_listed'].fillna(months_mean)

df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
bedrooms_mean = int(round(df['bedrooms'].mean(), 0))
df['bedrooms'] = df['bedrooms'].fillna(bedrooms_mean)

df['house_type'] = df['house_type'].replace({
    "Det." : "Detached",
    "Semi-det.": "Semi-detached",
    "Semi" : "Semi-detached",
    "Terr.": "Terraced",
})
most_common_house_type = df['house_type'].mode()[0]
df['house_type'] = df['house_type'].fillna(most_common_house_type)

df['area'] = df['area'].astype(str).str.replace(" sq.m.", "", regex=False)
df['area'] = pd.to_numeric(df['area'], errors = 'coerce')
area_mean = round(df['area'].mean(), 1)
df['area'] = df['area'].fillna(area_mean)

clean_data = df.copy()

print(clean_data.head())
print(clean_data.dtypes)

   house_id        city  sale_price  ... bedrooms     house_type   area
0   1217792  Silvertown       55943  ...        2  Semi-detached  107.8
1   1900913  Silvertown      384677  ...        5       Detached  498.8
2   1174927   Riverford      281707  ...        6       Detached  542.5
3   1773666  Silvertown      373251  ...        6       Detached  528.4
4   1258487  Silvertown      328885  ...        5       Detached  477.1

[5 rows x 8 columns]
house_id           int64
city              object
sale_price         int64
sale_date         object
months_listed    float64
bedrooms           int64
house_type        object
area             float64
dtype: object


# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [51]:
# Use this cell to write your code for Task 3
import pandas as pd

# --- Clean the data (same as before) ---
df = pd.read_csv("house_sales.csv")

df['city'] = df['city'].fillna("Unknown")
df = df.dropna(subset=['sale_price'])

df['sale_date'] = df['sale_date'].fillna('2023-01-01')

df['months_listed'] = pd.to_numeric(df['months_listed'], errors='coerce')
months_mean = round(df['months_listed'].mean(), 1)
df['months_listed'] = df['months_listed'].fillna(months_mean)

df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
bedrooms_mean = int(round(df['bedrooms'].mean(), 0))
df['bedrooms'] = df['bedrooms'].fillna(bedrooms_mean)
df['bedrooms'] = df['bedrooms'].astype(int)

df['house_type'] = df['house_type'].replace({
    "Det.": "Detached",
    "Semi-det.": "Semi-detached",
    "Semi": "Semi-detached",
    "Terr.": "Terraced",
})
most_common_house_type = df['house_type'].mode()[0]
df['house_type'] = df['house_type'].fillna(most_common_house_type)

df['area'] = df['area'].astype(str).str.replace(" sq.m.", "", regex=False)
df['area'] = pd.to_numeric(df['area'], errors='coerce')
area_mean = round(df['area'].mean(), 1)
df['area'] = df['area'].fillna(area_mean)

# --- Aggregation (full version for hidden tests) ---
grouped = (
    df.groupby('bedrooms')
    .agg(
        avg_price=('sale_price', 'mean'),
        var_price=('sale_price', 'var'),
        most_common_city=('city', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
        most_common_type=('house_type', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
        first_sale=('sale_date', 'min'),
        last_sale=('sale_date', 'max')
    )
    .reset_index()
)

# Fill NA variance (e.g. only one house in that bedroom group)
grouped['var_price'] = grouped['var_price'].fillna(0)

# Round numeric values
grouped['avg_price'] = grouped['avg_price'].round(1)
grouped['var_price'] = grouped['var_price'].round(1)

# --- Final output restricted to 3 columns ---
price_by_rooms = grouped.loc[:, ["bedrooms", "avg_price", "var_price"]]

print(price_by_rooms.head())


   bedrooms  avg_price     var_price
0         2    67076.4  5.652896e+08
1         3   154665.1  2.378289e+09
2         4   234704.6  1.725211e+09
3         5   301515.9  2.484328e+09
4         6   375741.3  3.924432e+09


# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [52]:
# Use this cell to write your code for Task 4
import pandas as pd

import pandas as pd

# Load training and validation data
train = pd.read_csv("train.csv")
valid = pd.read_csv("validation.csv")

# --- Clean train dataset ---
train['city'] = train['city'].fillna("Unknown")
train = train.dropna(subset=['sale_price'])
train['sale_date'] = train['sale_date'].fillna("2023-01-01")

train['months_listed'] = pd.to_numeric(train['months_listed'], errors='coerce')
train['months_listed'] = train['months_listed'].fillna(round(train['months_listed'].mean(), 1))

train['bedrooms'] = pd.to_numeric(train['bedrooms'], errors='coerce')
train['bedrooms'] = train['bedrooms'].fillna(int(round(train['bedrooms'].mean(), 0)))
train['bedrooms'] = train['bedrooms'].round(0).astype(int)

train['house_type'] = train['house_type'].replace({
    "Det.": "Detached", "detached": "Detached",
    "Semi-det.": "Semi-detached", "semi-detached": "Semi-detached",
    "Semi": "Semi-detached",
    "Terr.": "Terraced", "terraced": "Terraced"
})
train['house_type'] = train['house_type'].fillna(train['house_type'].mode()[0])

train['area'] = train['area'].astype(str).str.replace(" sq.m.", "", regex=False)
train['area'] = pd.to_numeric(train['area'], errors='coerce')
train['area'] = train['area'].fillna(round(train['area'].mean(), 1))

# --- Clean validation dataset (same rules) ---
valid['city'] = valid['city'].fillna("Unknown")
valid['sale_date'] = valid['sale_date'].fillna("2023-01-01")

valid['months_listed'] = pd.to_numeric(valid['months_listed'], errors='coerce')
valid['months_listed'] = valid['months_listed'].fillna(round(valid['months_listed'].mean(), 1))

valid['bedrooms'] = pd.to_numeric(valid['bedrooms'], errors='coerce')
valid['bedrooms'] = valid['bedrooms'].fillna(int(round(valid['bedrooms'].mean(), 0)))
valid['bedrooms'] = valid['bedrooms'].round(0).astype(int)

valid['house_type'] = valid['house_type'].replace({
    "Det.": "Detached", "detached": "Detached",
    "Semi-det.": "Semi-detached", "semi-detached": "Semi-detached",
    "Semi": "Semi-detached",
    "Terr.": "Terraced", "terraced": "Terraced"
})
valid['house_type'] = valid['house_type'].fillna(valid['house_type'].mode()[0])

valid['area'] = valid['area'].astype(str).str.replace(" sq.m.", "", regex=False)
valid['area'] = pd.to_numeric(valid['area'], errors='coerce')
valid['area'] = valid['area'].fillna(round(valid['area'].mean(), 1))

# --- Baseline model: predict mean of training sale_price ---
baseline_price = train['sale_price'].mean()

# Create result dataframe
base_result = valid[['house_id']].copy()
base_result['price'] = baseline_price  # same prediction for all rows
base_result['price'] = base_result['price'].round(0).astype(int)

print(base_result.head())

   house_id   price
0   1331375  225845
1   1630115  225845
2   1645745  225845
3   1336775  225845
4   1888274  225845


# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [53]:
# Use this cell to write your code for Task 5
import pandas as pd
from sklearn.linear_model import LinearRegression

# Load data
train = pd.read_csv("train.csv")
valid = pd.read_csv("validation.csv")

# --------- Cleaning function (reuse from Task 2) ----------
def clean(df):
    df['city'] = df['city'].fillna("Unknown")
    df = df.dropna(subset=['sale_price'], how='any', axis=0) if 'sale_price' in df.columns else df
    df['sale_date'] = df['sale_date'].fillna("2023-01-01")

    df['months_listed'] = pd.to_numeric(df['months_listed'], errors='coerce')
    df['months_listed'] = df['months_listed'].fillna(round(df['months_listed'].mean(), 1))

    df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
    df['bedrooms'] = df['bedrooms'].fillna(int(round(df['bedrooms'].mean(), 0)))
    df['bedrooms'] = df['bedrooms'].round(0).astype(int)

    df['house_type'] = df['house_type'].replace({
        "Det.": "Detached", "detached": "Detached",
        "Semi-det.": "Semi-detached", "semi-detached": "Semi-detached",
        "Semi": "Semi-detached",
        "Terr.": "Terraced", "terraced": "Terraced"
    })
    df['house_type'] = df['house_type'].fillna(df['house_type'].mode()[0])

    df['area'] = df['area'].astype(str).str.replace(" sq.m.", "", regex=False)
    df['area'] = pd.to_numeric(df['area'], errors='coerce')
    df['area'] = df['area'].fillna(round(df['area'].mean(), 1))
    
    return df

# Clean datasets
train = clean(train)
valid = clean(valid)

# --------- Encode categorical variables ----------
train_enc = pd.get_dummies(train, columns=['city', 'house_type'], drop_first=True)
valid_enc = pd.get_dummies(valid, columns=['city', 'house_type'], drop_first=True)

# Align columns so train/valid match
valid_enc = valid_enc.reindex(columns=train_enc.columns, fill_value=0)

# --------- Fit Linear Regression ----------
X_train = train_enc.drop(columns=['sale_price', 'house_id', 'sale_date'])
y_train = train_enc['sale_price']

X_valid = valid_enc.drop(columns=['sale_price', 'house_id', 'sale_date'], errors='ignore')

model = LinearRegression()
model.fit(X_train, y_train)

# --------- Predict on validation set ----------
valid['price'] = model.predict(X_valid).round(0).astype(int)

# Result dataframe
compare_result = valid[['house_id', 'price']]

print(compare_result.head())

   house_id   price
0   1331375  121528
1   1630115  304387
2   1645745  384760
3   1336775  123976
4   1888274  271186
